# Capybara RC3 — Integrated Preservation + Expansion Deployment

RC3 upgrades the still-live Alpha documentation **in place** into the Original Edition 1.0.0. It recovers the exact public source/assets before any push, retains the Alpha visual language as the primary presentation, integrates expanded guidance into the same routes, adds genuinely new sections to the same navigation, audits completeness, pushes `main`, rebuilds the existing Read the Docs project, and verifies protected visual behaviors after publication.


In [1]:
from pathlib import Path
import subprocess, sys, json, time, getpass, requests
REPO_ROOT=Path.cwd().resolve()
if not (REPO_ROOT/'.readthedocs.yaml').exists():
    raise RuntimeError('Open/run this notebook from the RC3 repository root.')
GITHUB_OWNER='BrianBowers-NapaCounty'
GITHUB_REPO='itam-itsm_integration_framework'
GITHUB_URL=f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
RTD_PROJECT_SLUG='capybara-framework'
RTD_VERSION='latest'
print('RC3 repository:',REPO_ROOT)
print('GitHub:',GITHUB_URL)
print('RTD:',RTD_PROJECT_SLUG)


RC3 repository: C:\Users\BBOWERS\Jupyter Notebooks\Capybara_Framework_GitHub_Repository_RC3_Integrated
GitHub: https://github.com/BrianBowers-NapaCounty/itam-itsm_integration_framework.git
RTD: capybara-framework


## 1. Recover Alpha and integrate RC3

Run this **before** the old public build is replaced. If an open Selenium `driver` exists in this kernel, the finalizer uses it as a fallback for public-file retrieval.


In [2]:
sys.path.insert(0,str(REPO_ROOT/'tools'))
from finalize_rc3 import finalize
report=finalize(REPO_ROOT,driver=globals().get('driver'))
if report['errors']:
    print(json.dumps(report['errors'][:30],indent=2))
    raise RuntimeError(f"RC3 finalization has {len(report['errors'])} recovery error(s). Nothing will be pushed.")
print('✓ Exact live Alpha baseline recovered and integrated.')


SOURCE 01/59: branding.rst.txt
SOURCE 02/59: changelog.md.txt
SOURCE 03/59: code-of-conduct.md.txt
SOURCE 04/59: contributing.md.txt
SOURCE 05/59: editions/alpha/0-overview.md.txt
SOURCE 06/59: editions/alpha/0-overview/0-1-introduction.md.txt
SOURCE 07/59: editions/alpha/0-overview/0-2-problem-statement.md.txt
SOURCE 08/59: editions/alpha/0-overview/0-3-executive-summary.md.txt
SOURCE 09/59: editions/alpha/0-overview/README.md.txt
SOURCE 10/59: editions/alpha/1-implementation-guide.md.txt
SOURCE 11/59: editions/alpha/1-implementation-guide/1-1-first-steps/1-1-1-prerequisites.md.txt
SOURCE 12/59: editions/alpha/1-implementation-guide/1-1-first-steps/1-1-2-budget-procurement.md.txt
SOURCE 13/59: editions/alpha/1-implementation-guide/1-1-first-steps/1-1-3-deployment-steps.md.txt
SOURCE 14/59: editions/alpha/1-implementation-guide/1-1-first-steps/1-1-4-configuration.md.txt
SOURCE 15/59: editions/alpha/1-implementation-guide/1-1-first-steps/README.md.txt
SOURCE 16/59: editions/alpha/1-impl

In [3]:
audit=subprocess.run([
    sys.executable,str(REPO_ROOT/'tools/audit_rc3.py'),
    '--repo',str(REPO_ROOT),'--strict'
],text=True)
if audit.returncode!=0:
    raise RuntimeError('RC3 completeness audit failed. Nothing will be pushed.')
print('✓ RC3 completeness audit PASS')
print('Evidence: legacy_baseline/RC3_COVERAGE_MATRIX.csv')


RuntimeError: RC3 completeness audit failed. Nothing will be pushed.

## 2. Commit and push the audited integrated source


In [ ]:
def git(*args,check=True):
    r=subprocess.run(['git','-C',str(REPO_ROOT),*args],capture_output=True,text=True)
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    if check and r.returncode:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stderr}")
    return r

if git('rev-parse','--git-dir',check=False).returncode!=0:
    git('init')
remotes=git('remote',check=False).stdout.split()
if 'origin' in remotes:
    git('remote','set-url','origin',GITHUB_URL)
else:
    git('remote','add','origin',GITHUB_URL)

git('add','-A')
if git('rev-parse','--verify','HEAD',check=False).returncode!=0:
    git('commit','-m','Publish Capybara Original Edition 1.0.0 RC3')
elif git('status','--porcelain',check=False).stdout.strip():
    git('commit','-m','Integrate complete Alpha baseline and expanded RC3 documentation')

git('branch','-M','main')
confirm=input(f'Type PUSH {GITHUB_OWNER}/{GITHUB_REPO} to publish the audited RC3 source: ').strip()
if confirm!=f'PUSH {GITHUB_OWNER}/{GITHUB_REPO}':
    raise RuntimeError('Push cancelled.')
git('push','-u','origin','main')
verify=git('ls-remote','--heads','origin','refs/heads/main')
if not verify.stdout.strip():
    raise RuntimeError('GitHub does not advertise refs/heads/main after push.')
print('✓ GitHub main:',verify.stdout.split()[0])


fatal: not a git repository (or any of the parent directories): .git
Initialized empty Git repository in C:/Users/BBOWERS/Jupyter Notebooks/Capybara_Framework_GitHub_Repository_RC3_Integrated/.git/
fatal: Needed a single revision
[master (root-commit) 59788fa] Publish Capybara Original Edition 1.0.0 RC3
 1213 files changed, 36218 insertions(+)
 create mode 100644 .gitattributes
 create mode 100644 .github/CODEOWNERS
 create mode 100644 .github/ISSUE_TEMPLATE/bug.yml
 create mode 100644 .github/ISSUE_TEMPLATE/enhancement.yml
 create mode 100644 .github/pull_request_template.md
 create mode 100644 .github/workflows/docs.yml
 create mode 100644 .github/workflows/release-artifacts.yml
 create mode 100644 .gitignore
 create mode 100644 .readthedocs.yaml
 create mode 100644 CHANGELOG.md
 create mode 100644 CITATION.cff
 create mode 100644 CODE_OF_CONDUCT.md
 create mode 100644 CONTRIBUTING.md
 create mode 100644 Capybara_RC3_Integrated_Deployment.ipynb
 create mode 100644 KNOWN_CONTENT_RECON

## 3. Reconnect, synchronize, and build the existing Read the Docs project


In [ ]:
RTD_API_TOKEN=globals().get('RTD_API_TOKEN','').strip()
if not RTD_API_TOKEN:
    RTD_API_TOKEN=getpass.getpass('Read the Docs API token (hidden): ').strip()
    globals()['RTD_API_TOKEN']=RTD_API_TOKEN
if not RTD_API_TOKEN:
    raise RuntimeError('No RTD API token supplied.')
API='https://app.readthedocs.org/api/v3'
def rtd(method,path,body=None,allowed=(200,201,202,204)):
    url=API.rstrip('/')+'/'+path.lstrip('/')
    headers={'Accept':'application/json','Authorization':f'Token {RTD_API_TOKEN}'}
    if body is not None: headers['Content-Type']='application/json'
    r=requests.request(method.upper(),url,headers=headers,json=body,timeout=60)
    try: data=r.json() if r.text else None
    except Exception: data=r.text
    if r.status_code not in allowed:
        raise RuntimeError({'status':r.status_code,'url':url,'data':data})
    return r.status_code,data

_,project=rtd('GET',f'projects/{RTD_PROJECT_SLUG}/')
print('Current RTD repository:',project.get('repository',{}).get('url'))
confirm=input(f'Type BUILD {RTD_PROJECT_SLUG} to reconnect/sync/build latest: ').strip()
if confirm!=f'BUILD {RTD_PROJECT_SLUG}':
    raise RuntimeError('RTD build cancelled.')
rtd('PATCH',f'projects/{RTD_PROJECT_SLUG}/',{
    'repository':{'url':GITHUB_URL.removesuffix('.git'),'type':'git'},
    'default_branch':'main','default_version':'latest',
    'readthedocs_yaml_path':'.readthedocs.yaml'})
rtd('POST',f'projects/{RTD_PROJECT_SLUG}/sync-versions/')
time.sleep(5)
_,version=rtd('GET',f'projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/')
if not version.get('active') or version.get('hidden'):
    rtd('PATCH',f'projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/',{'active':True,'hidden':False})
rtd('POST',f'projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/builds/')
print('✓ Build requested')


In [ ]:
def build_state(b):
    s=b.get('state')
    return (s.get('code') or s.get('name')) if isinstance(s,dict) else str(s or '')

build_id=None
for _ in range(40):
    _,data=rtd('GET',f'projects/{RTD_PROJECT_SLUG}/builds/?limit=20')
    for b in data.get('results',[]):
        v=b.get('version'); v=v.get('slug') if isinstance(v,dict) else v
        if v==RTD_VERSION:
            build_id=b.get('id'); break
    if build_id: break
    time.sleep(2)
if not build_id:
    raise RuntimeError('Could not identify the latest RTD build.')
last=None
for _ in range(180):
    _,b=rtd('GET',f'projects/{RTD_PROJECT_SLUG}/builds/{build_id}/?expand=config')
    state=build_state(b).lower()
    if state!=last:
        print(time.strftime('%H:%M:%S'),state,'success=',b.get('success'))
        last=state
    if state in ('finished','cancelled'): break
    time.sleep(5)
else:
    raise RuntimeError('RTD build polling timed out.')
if not b.get('success'):
    if globals().get('driver'):
        driver.get(f'https://app.readthedocs.org/projects/{RTD_PROJECT_SLUG}/builds/{build_id}/')
    raise RuntimeError(f'RTD build {build_id} failed; build page opened if Selenium is available.')
print('✓ RTD build succeeded:',build_id,'commit=',b.get('commit'))


## 4. Verify the published look and protected media behavior


In [ ]:
docs_url=f'https://{RTD_PROJECT_SLUG}.readthedocs.io/en/latest/'
if globals().get('driver'):
    from verify_live_visual_behavior import verify
    verify(driver,REPO_ROOT)
    driver.get(docs_url)
else:
    print('No Selenium driver in this kernel; browser-rendered visual audit skipped.')
    print('Run verify_live_visual_behavior.verify(driver, REPO_ROOT) before declaring RC3 final.')
print('LIVE:',docs_url)


In [ ]:
# =====================================================================
# RC3 LOCAL-ONLY RECONCILIATION REPAIR
# NO REDOWNLOADS — uses the Alpha files you already recovered.
# =====================================================================

from pathlib import Path
import json, re, shutil, subprocess, sys

REPO_ROOT = Path(globals().get("REPO_ROOT", Path.cwd())).resolve()
DOCS = REPO_ROOT / "docs"
SEED = REPO_ROOT / "rc3_seed" / "docs"
ARCHIVE = DOCS / "_legacy_original_source"
MANIFEST = REPO_ROOT / "legacy_baseline" / "LEGACY_BASELINE_MANIFEST.json"
AUDIT_JSON = REPO_ROOT / "legacy_baseline" / "RC3_COMPLETENESS_AUDIT.json"

sys.path.insert(0, str(REPO_ROOT / "tools"))

from finalize_rc3 import (
    authored,
    MD_TOC,
    RST_TOC,
    parse_md_toc,
    parse_rst_toc,
    seed_body_md,
    seed_body_rst,
    norm,
    add_style_notes_links,
)

print("=" * 72)
print("RC3 LOCAL RECONCILIATION")
print("=" * 72)

if not ARCHIVE.exists():
    raise RuntimeError(
        "docs/_legacy_original_source is missing. "
        "The Alpha recovery has not completed."
    )

manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))

# Show exactly what the previous audit complained about.
if AUDIT_JSON.exists():
    old = json.loads(AUDIT_JSON.read_text(encoding="utf-8"))

    print(
        f"\nPrevious audit: {old.get('status')} — "
        f"{len(old.get('errors', []))} error(s)"
    )

    for err in old.get("errors", []):
        print("  •", err)

# Pages for which the currently-live treatment is already canonical.
EXACT_LIVE = {
    "branding.rst",
    "media-kit.rst",
    "editions/alpha/5-visual-resources/5-3-videos.rst",
    "editions/alpha/5-visual-resources/5-4-graphics.rst",
    "editions/alpha/5-visual-resources/5-5-icons.rst",
}

def all_md_toc_items(text):
    out = []
    for m in MD_TOC.finditer(text):
        _, items = parse_md_toc(m.group(1))
        out.extend(items)
    return out

def all_rst_toc_items(text):
    out = []
    for m in RST_TOC.finditer(text):
        _, items = parse_rst_toc(m.group(1))
        out.extend(items)
    return out

def compose_md(legacy, seed):
    """
    Keep the recovered Alpha page literally intact.
    Append ONLY new RC3 navigation/content after it.
    """
    out = legacy.rstrip()

    legacy_items = set(all_md_toc_items(legacy))
    new_items = [
        x for x in all_md_toc_items(seed)
        if x not in legacy_items
    ]

    if new_items:
        out += (
            "\n\n---\n\n"
            "## Additional Original Edition Topics\n\n"
            "```{toctree}\n"
            ":maxdepth: 3\n"
            ":caption: Expanded 1.0.0 Documentation\n\n"
            + "\n".join(new_items)
            + "\n```\n"
        )

    body = seed_body_md(seed)

    if body and norm(body) not in norm(legacy):
        out += (
            '\n\n<div class="rc3-extension">\n\n'
            "## Expanded 1.0.0 Guidance\n\n"
            + body
            + "\n\n</div>\n"
        )

    return out.rstrip() + "\n"

def compose_rst(legacy, seed):
    """
    Same policy for reStructuredText:
    exact Alpha source first, new material afterward.
    """
    out = legacy.rstrip()

    legacy_items = set(all_rst_toc_items(legacy))
    new_items = [
        x for x in all_rst_toc_items(seed)
        if x not in legacy_items
    ]

    if new_items:
        out += (
            "\n\nAdditional Original Edition Topics\n"
            "-----------------------------------\n\n"
            ".. toctree::\n"
            "   :maxdepth: 3\n"
            "   :caption: Expanded 1.0.0 Documentation\n\n"
            + "\n".join("   " + x for x in new_items)
            + "\n"
        )

    body = seed_body_rst(seed)

    if body and norm(body) not in norm(legacy):
        out += (
            "\n\n.. raw:: html\n\n"
            '   <div class="rc3-extension">\n\n'
            "Expanded 1.0.0 Guidance\n"
            "-----------------------\n\n"
            + body
            + "\n\n.. raw:: html\n\n"
            "   </div>\n"
        )

    return out.rstrip() + "\n"

# ---------------------------------------------------------------------
# REBUILD EVERY EXISTING PAGE FROM THE EXACT DOWNLOADED ALPHA SOURCE
# ---------------------------------------------------------------------

legacy_docnames = set()
rebuilt = 0

for src in manifest["sources"]:

    rel = authored(src)

    legacy_docnames.add(
        Path(rel).with_suffix("").as_posix()
    )

    archived = ARCHIVE / src

    if not archived.exists():
        raise RuntimeError(
            f"Recovered Alpha source unexpectedly missing: {src}"
        )

    legacy = archived.read_text(
        encoding="utf-8",
        errors="replace",
    )

    target = DOCS / rel

    # Locate RC3 treatment of the same logical page.
    seed_target = SEED / rel

    if not seed_target.exists():
        alt_seed = seed_target.with_suffix(
            ".rst" if seed_target.suffix == ".md" else ".md"
        )

        if alt_seed.exists():
            seed_target = alt_seed

    seed = (
        seed_target.read_text(
            encoding="utf-8",
            errors="replace",
        )
        if seed_target.exists()
        else ""
    )

    # Do not leave competing .md/.rst versions of one Alpha docname.
    alternate = target.with_suffix(
        ".rst" if target.suffix == ".md" else ".md"
    )

    if alternate.exists() and alternate != target:
        alternate.unlink()

    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if rel in EXACT_LIVE or not seed:
        outgoing = legacy

    elif rel.endswith(".md"):
        outgoing = compose_md(
            legacy,
            seed,
        )

    elif rel.endswith(".rst"):
        outgoing = compose_rst(
            legacy,
            seed,
        )

    else:
        outgoing = legacy

    target.write_text(
        outgoing,
        encoding="utf-8",
    )

    rebuilt += 1

# ---------------------------------------------------------------------
# COPY ONLY GENUINELY NEW RC3 PAGES
# ---------------------------------------------------------------------

new_pages = 0

for source in SEED.rglob("*"):

    if not source.is_file():
        continue

    rel = source.relative_to(SEED)

    # Alpha assets already came from the live-site recovery.
    if rel.parts and rel.parts[0] in (
        "_images",
        "_static",
    ):
        continue

    target = DOCS / rel

    if source.suffix.lower() in (
        ".md",
        ".rst",
    ):
        docname = rel.with_suffix("").as_posix()

        # Existing Alpha document: already rebuilt above.
        if docname in legacy_docnames:
            continue

        target.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        shutil.copy2(
            source,
            target,
        )

        new_pages += 1

    elif not target.exists():
        target.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        shutil.copy2(
            source,
            target,
        )

# ---------------------------------------------------------------------
# REQUIRED CROSS-LINKS / ROOT CONTRIBUTING
# ---------------------------------------------------------------------

add_style_notes_links(
    DOCS / "style-notes.md"
)

if (DOCS / "contributing.md").exists():
    shutil.copy2(
        DOCS / "contributing.md",
        REPO_ROOT / "CONTRIBUTING.md",
    )

print()
print(f"✓ Rebuilt {rebuilt} Alpha pages from exact downloaded source")
print(f"✓ Added {new_pages} genuinely new RC3 pages")
print("✓ No network requests were made")

# ---------------------------------------------------------------------
# RERUN THE REAL COMPLETENESS AUDIT
# ---------------------------------------------------------------------

audit = subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "tools" / "audit_rc3.py"),
        "--repo",
        str(REPO_ROOT),
        "--strict",
    ],
    capture_output=True,
    text=True,
)

print()
print(audit.stdout)

if audit.stderr.strip():
    print("AUDIT STDERR:")
    print(audit.stderr)

if audit.returncode != 0:

    latest = json.loads(
        AUDIT_JSON.read_text(
            encoding="utf-8"
        )
    )

    print("\nREMAINING GENUINE FAILURES:")

    for err in latest.get("errors", []):
        print("  •", err)

    raise RuntimeError(
        "RC3 still has a genuine completeness failure. "
        "Do NOT rerun the download phase; the exact remaining "
        "problem(s) are printed directly above."
    )

print("=" * 72)
print("✓ RC3 COMPLETENESS AUDIT PASS")
print("=" * 72)
print()
print("Continue directly with the GitHub PUSH cell.")